# Feature extraction

This notebooks contains the code used to extract Librosa, RQA and CLAP features for the MSc thesis Human vs AI Jazz.

### Librosa

In [ ]:
import librosa
import numpy as np
import scipy
import os
from pathlib import Path
from tqdm import tqdm

In [ ]:
# ============================
# FEATURE EXTRACTION FUNCTION
# ============================
def extract_features(
    audio_path,
    sr=22050,
    hop_length=512,
    n_fft=2048,
    duration=None):  # e.g. 60 for 60 seconds, or None for full track

    y, sr = librosa.load(audio_path, sr=sr, duration=duration)


    f0 = librosa.yin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr,frame_length=n_fft, hop_length=hop_length)

    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length)

    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)

    tempogram = librosa.feature.tempogram(y=y, sr=sr, onset_envelope=onset_env, hop_length=hop_length)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_mfcc=20, n_fft = n_fft)

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length)

    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length)

    rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length)

    features = {
        "f0": f0,
        "chroma": chroma,
        "onset_env": onset_env,
        "tempogram": tempogram,
        "mfcc": mfcc,
        "spectral_centroid": spectral_centroid,
        "spectral_contrast": spectral_contrast,
        "rms": rms
        }

    return features

In [ ]:
# ==============================
# MAIN FEATURE EXTRACTION SCRIPT
# ==============================

# Root folder containing:
# FMA_final/
# jazzset_final/
# suno_final/
# udio_final/
input_root = #

# Output folder
output_root = #
output_root.mkdir(parents=True, exist_ok=True)

mp3_files = list(input_root.rglob("*.mp3"))
print(f"Found {len(mp3_files)} mp3 files.")

for mp3_path in tqdm(mp3_files, desc="Extracting features"):

    try:
        features = extract_features(mp3_path)

        relative_path = mp3_path.relative_to(input_root)

        output_file = output_root / relative_path.with_suffix(".npz")

        output_file.parent.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(output_file, **features)

    except Exception as e:
        print(f"\nFailed on: {mp3_path}")
        print(e)

print("Done.")

In [ ]:
# =========================
# CREATING SUMMARY FEATURES
# =========================

# Root folder of .npz files
results_root = #
summary_path = #


group_map = {
    "FMA_final": "human",
    "jazzset_final": "human",
    "suno_final": "ai",
    "udio_final": "ai"
}


# Extracting summary features

rows = []

npz_files = list(results_root.rglob("*.npz"))

for npz_path in tqdm(npz_files):

    try:
        data = np.load(npz_path)

        source = npz_path.parts[-2]
        group = group_map[source]

        row = {
            "song": npz_path.stem,
            "source": source,
            "group": group
        }


        # F0
        f0 = data["f0"]

        row["f0_mean"] = np.nanmean(f0)
        row["f0_std"] = np.nanstd(f0)


        # RMS
        rms = data["rms"].squeeze()

        row["rms_mean"] = rms.mean()
        row["rms_std"] = rms.std()


        # Spectral centroid

        sc = data["spectral_centroid"].squeeze()

        row["sc_mean"] = sc.mean()
        row["sc_std"] = sc.std()


        # Spectral contrast
        contrast = data["spectral_contrast"]

        for i in range(contrast.shape[0]):
            row[f"contrast_{i}_mean"] = contrast[i].mean()
            row[f"contrast_{i}_std"] = contrast[i].std()


        # MFCC
        mfcc = data["mfcc"]

        for i in range(mfcc.shape[0]):
            row[f"mfcc_{i+1}_mean"] = mfcc[i].mean()
            row[f"mfcc_{i+1}_std"] = mfcc[i].std()


        # Chroma
        chroma = data["chroma"]

        for i in range(12):
            row[f"chroma_{i}_mean"] = chroma[i].mean()


        # Tempogram
        tempogram = data["tempogram"]

        for i in range(tempogram.shape[0]):
            row[f"tempogram_{i}_mean"] = tempogram[i].mean()


        # Onset envelope
        onset = data["onset_env"]

        row["onset_mean"] = onset.mean()
        row["onset_std"] = onset.std()

        rows.append(row)

    except Exception as e:
        print(f"Failed: {npz_path}")
        print(e)


# Dataframe
df = pd.DataFrame(rows)

df.to_csv(summary_path, index=False)

### RQA metrics

In [ ]:
!pip install pyrqa

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.spatial.distance import pdist

from pyrqa.time_series import TimeSeries
from pyrqa.time_series import EmbeddedSeries
from pyrqa.settings import Settings
from pyrqa.analysis_type import Classic
from pyrqa.metric import EuclideanMetric
from pyrqa.neighbourhood import FixedRadius
from pyrqa.computation import RQAComputation
from scipy.signal import decimate

In [ ]:
# CONFIG

ROOT_DIR = # directory with the full .npz files from the Librosa feature extraction

FOLDERS = ["FMA_final", "jazzset_final", "suno_final", "udio_final"]

SCALAR_FEATURES = ["f0", "rms", "spectral_centroid", "onset_env"]
VECTOR_FEATURES = ["mfcc", "chroma", "spectral_contrast", "tempogram"]

DOWNSAMPLE = 4
TARGET_RR = 2

WINDOW_SEC = 30
WINDOW_OVERLAP = 0.5
FEATURE_SR = 22050 / 512

SCALAR_EMBED_PARAMS = { # parameters estimated through AMI and FNN
    "f0": {"tau": 5, "m": 6},
    "rms": {"tau": 3, "m": 4},
    "spectral_centroid": {"tau": 3, "m": 4},
    "onset_env": {"tau": 3, "m": 4}
}



# HELPERS

def zscore(x, axis=-1):
    return (x - np.mean(x, axis=axis, keepdims=True)) / (np.std(x, axis=axis, keepdims=True) + 1e-12)


def make_embedding(x, dim=EMBED_DIM, delay=DELAY):
    n = len(x) - (dim - 1) * delay
    emb = np.empty((n, dim))
    for i in range(dim):
        emb[:, i] = x[i * delay:i * delay + n]
    return emb


def fast_epsilon(X, rr=0.02, sample_size=2000):
    n = len(X)
    idx = np.random.choice(n, size=min(n, sample_size), replace=False)
    d = pdist(X[idx], metric="euclidean")
    return np.quantile(d, rr)


def sliding_windows(x, window_size, step_size):
    n = len(x)
    windows = []
    start = 0

    while start < n:
        end = min(start + window_size, n)
        windows.append(x[start:end])
        if end == n:
            break
        start += step_size

    return windows


# MAIN RQA FUNCTION

def compute_rqa(X, embedding_dimension=EMBED_DIM, time_delay=DELAY):

    X = np.asarray(X).astype(np.float64)
    X = np.nan_to_num(X)

    if DOWNSAMPLE > 1:

        if X.ndim == 1:

            if len(X) > 100:
                X = decimate(
                    X,
                    DOWNSAMPLE,
                    ftype='fir',
                    zero_phase=True
                )

        elif X.ndim == 2:

            if X.shape[0] < X.shape[1]:
                X = X.T

            if len(X) > 100:
                X = decimate(
                    X,
                    DOWNSAMPLE,
                    axis=0,
                    ftype='fir',
                    zero_phase=True
                )

    eps = None

    if X.ndim == 2:

        if X.shape[0] < X.shape[1]:
            X = X.T

        X = zscore(X, axis=0)

        ts = EmbeddedSeries(X)
        eps = fast_epsilon(X)

    else:

        X = np.squeeze(X)
        X = zscore(X)

        embedded = make_embedding(X, embedding_dimension, time_delay)

        ts = TimeSeries(
            X,
            embedding_dimension=embedding_dimension,
            time_delay=time_delay
        )

        eps = fast_epsilon(embedded)

    settings = Settings(
        ts,
        analysis_type=Classic,
        neighbourhood=FixedRadius(eps),
        similarity_measure=EuclideanMetric,
        theiler_corrector=1
    )

    res = RQAComputation.create(settings).run()

    return eps, res


# MAIN PIPELINE

rows = []

for folder in FOLDERS:

    path = os.path.join(ROOT_DIR, folder)
    files = glob.glob(os.path.join(path, "*.npz"))

    source = folder.replace("_final", "")
    group = "human" if source in ["FMA", "jazzset"] else "ai"

    print("\n" + "=" * 70)
    print(f"Processing {folder} | {len(files)} tracks")
    print("=" * 70)

    for f in tqdm(files, desc=folder):

        track_id = os.path.basename(f).replace(".npz", "")
        data = np.load(f, allow_pickle=True)

        row = {
            "filename": track_id,
            "group": group,
            "source": source
        }

        # SCALAR FEATURES

        for feat in SCALAR_FEATURES:

            if feat not in data.files:
                continue

            try:
                x = np.squeeze(data[feat])

                window_size = int(WINDOW_SEC * FEATURE_SR)
                step_size = int(window_size * (1 - WINDOW_OVERLAP))
                windows = sliding_windows(x, window_size, step_size)

                metrics = {
                    "DET": [],
                    "ENTR": [],
                    "LAM": [],
                    "DIV": [],
                    "Lmax": [],
                    "TT": [],
                    "RR": []
                }

                eps_list = []

                params = SCALAR_EMBED_PARAMS.get(feat, {"m": EMBED_DIM, "tau": DELAY})

                for w in windows:

                    eps, res = compute_rqa(
                        w,
                        embedding_dimension=params["m"],
                        time_delay=params["tau"]
                    )

                    eps_list.append(eps)

                    metrics["DET"].append(res.determinism)
                    metrics["ENTR"].append(res.entropy_diagonal_lines)
                    metrics["LAM"].append(res.laminarity)
                    metrics["DIV"].append(res.divergence)
                    metrics["Lmax"].append(res.longest_diagonal_line)
                    metrics["TT"].append(res.trapping_time)
                    metrics["RR"].append(res.recurrence_rate)

                if len(metrics["DET"]) > 0:
                    for k, v in metrics.items():
                        row[f"{k}_{feat}_mean"] = float(np.mean(v))
                        row[f"{k}_{feat}_median"] = float(np.median(v))
                        row[f"{k}_{feat}_std"] = float(np.std(v))

                if len(eps_list) > 0:
                    row[f"epsilon_{feat}"] = float(np.mean(eps_list))

            except Exception as e:
                print(f"Scalar error {feat} in {track_id}: {e}")


        # VECTOR FEATURES

        for feat in VECTOR_FEATURES:

            if feat not in data.files:
                continue

            try:
                X = data[feat]

                window_size = int(WINDOW_SEC * FEATURE_SR)
                step_size = int(window_size * (1 - WINDOW_OVERLAP))
                windows = sliding_windows(X, window_size, step_size)

                metrics = {
                    "DET": [],
                    "ENTR": [],
                    "LAM": [],
                    "DIV": [],
                    "Lmax": [],
                    "TT": [],
                    "RR": []
                }

                eps_list = []

                for w in windows:

                    eps, res = compute_rqa(w)

                    eps_list.append(eps)

                    metrics["DET"].append(res.determinism)
                    metrics["ENTR"].append(res.entropy_diagonal_lines)
                    metrics["LAM"].append(res.laminarity)
                    metrics["DIV"].append(res.divergence)
                    metrics["Lmax"].append(res.longest_diagonal_line)
                    metrics["TT"].append(res.trapping_time)
                    metrics["RR"].append(res.recurrence_rate)

                if len(metrics["DET"]) > 0:
                    for k, v in metrics.items():
                        row[f"{k}_{feat}_mean"] = float(np.mean(v))
                        row[f"{k}_{feat}_median"] = float(np.median(v))
                        row[f"{k}_{feat}_std"] = float(np.std(v))

                if len(eps_list) > 0:
                    row[f"epsilon_{feat}"] = float(np.mean(eps_list))

            except Exception as e:
                print(f"Vector error {feat} in {track_id}: {e}")


        # SAVE ROW

        rows.append(row)

        out_path = #

        pd.DataFrame([row]).to_csv(
            out_path,
            mode="a",
            header=not os.path.exists(out_path),
            index=False
        )


df = pd.DataFrame(rows)
df.to_csv('', index=False)

print("\nDONE")

### CLAP embeddings

In [ ]:
import os
import numpy as np
import librosa
from tqdm import tqdm
import torch
import laion_clap

In [ ]:
# CONFIG

DATA_DIR = # audio folder

SOURCES = {
    "FMA_final": 0,
    "jazzset_final": 1,
    "suno_final": 2,
    "udio_final": 3
}

SR = 48000

CHUNK_DURATION = 10
HOP_DURATION = 5

CHUNK_SIZE = SR * CHUNK_DURATION
HOP_SIZE = SR * HOP_DURATION

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# LOAD CLAP MODEL
model = laion_clap.CLAP_Module(
    enable_fusion=False,
    amodel='HTSAT-base'
)

model.load_ckpt(
    '/music_audioset_epoch_15_esc_90.14.pt'
)

model = model.to(device)
model.eval()

# FILE COLLECTION
def get_files(folder):
    path = os.path.join(DATA_DIR, folder)
    return [
        os.path.join(path, f)
        for f in os.listdir(path)
        if f.endswith(".mp3")
    ]

all_files = []
labels = []

for folder, label in SOURCES.items():
    files = get_files(folder)
    all_files.extend(files)
    labels.extend([label] * len(files))

print("Total files:", len(all_files))

# AUDIO LOADING
def load_audio(path):
    audio, _ = librosa.load(path, sr=SR, mono=True)
    return audio


# CHUNKING
def chunk_audio(audio, chunk_size, hop_size):
    chunks = []

    if len(audio) < chunk_size:
        return chunks

    for start in range(0, len(audio) - chunk_size + 1, hop_size):
        chunks.append(audio[start:start + chunk_size])

    return chunks


# EMBEDDING EXTRACTION

def extract_track_embedding(audio):

    chunks = chunk_audio(audio, CHUNK_SIZE, HOP_SIZE)

    if len(chunks) == 0:
        return None

    chunk_embeddings = []

    for chunk in chunks:

        tensor_audio = torch.tensor(chunk, dtype=torch.float32).to(device)

        with torch.no_grad():
            emb = model.get_audio_embedding_from_data(
                x=[tensor_audio],
                use_tensor=True
            )

        emb = emb.squeeze().detach().cpu().numpy()
        chunk_embeddings.append(emb)

    chunk_embeddings = np.array(chunk_embeddings)

    mean_emb = np.mean(chunk_embeddings, axis=0)
    std_emb = np.std(chunk_embeddings, axis=0)

    return np.concatenate([mean_emb, std_emb])


# MAIN LOOP

embeddings = []
used_labels = []
used_paths = []

for path, label in tqdm(list(zip(all_files, labels))):

    try:
        audio = load_audio(path)

        emb = extract_track_embedding(audio)

        if emb is None:
            continue

        embeddings.append(emb)
        used_labels.append(label)
        used_paths.append(path)

    except Exception as e:
        print(f"Failed on {path}: {e}")


X_all = np.array(embeddings)
y_all = np.array(used_labels)

np.savez(
    "",
    X=X_all,
    y=y_all,
    paths=np.array(used_paths)
)
